### **PART 1 - CONNECTING TO LASTFM API AND BRINGING IN NECESSARY DATA, PUTTING INTO POLISHED DATAFRAMES** ###

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('LASTFM_API_KEY')

In [ ]:
import requests
import pandas as pd
import json
from sklearn.cluster import KMeans

In [ ]:
url = "http://ws.audioscrobbler.com/2.0/"

In [ ]:
#Checking what this API end-point returns using Coldplay as an example.
#Our end goal is to create an artist recommendation system that is similar/better

params_similar_artists = {
    "method":"artist.getSimilar",
    "limit": 5,
    "artist": "Coldplay",
    "format":"json",
    "autocorrect": 1,
    "api_key": api_key
}

response = requests.get(url, params_similar_artists)

data_similar_artists = response.json()

data_similar_artists

In [ ]:
#Looking at tracks end-point just to get an idea of what the structure of the JSON data looks like

params_tracks = {
    "method": "chart.getTopTracks",
    "api_key": api_key,
    "format":"json",
    "limit": 10
}

response = requests.get(url, params_tracks)

data_tracks = response.json()

data_tracks

In [ ]:
#Getting the top 1000 artists on the website by playcount. Limiting our code to the 1000 most popular artists only 

params = {
    "method": "chart.gettopartists",
    "api_key": api_key,
    "format": "json",
    "limit": 1000
}

response = requests.get(url, params=params)
data = response.json()

print(data)

In [ ]:
#Readable human format so we can understand the JSON format better
print(json.dumps(data['artists']['artist'][0], indent=4))

In [ ]:
#Ensuring it is returning 1000 artists and not a different number

print(len(data['artists']['artist']))

In [ ]:
#Creating the df_artists and cleaning it so we keep the relevant columns and drop anything we don't need

df = pd.json_normalize(data['artists']['artist'])

df_artists = df[['name', 'playcount', 'listeners', 'url', 'streamable']]

df_artists

In [ ]:
#Creating a list of tags for each artist from the getTopTags end-point that we will later use to cluster artists

rows = []

for artist in df_artists['name']:

    params = {
        "method": "artist.getTopTags",
        "api_key": api_key,
        "format": "json",
        "artist": artist,
        "autocorrect": 1
    }

    response = requests.get(url, params=params)
    data_tags = response.json()

    tags = data_tags.get("toptags", {}).get("tag", [])

    if isinstance(tags, dict):
        tags = [tags]

    for t in tags:
        rows.append({
            "artist": artist,
            "tag": t["name"],
            "count": int(t["count"])
        })


In [ ]:
#Understanding the structure of the tags JSON

print(json.dumps(data_tags['toptags']['tag'], indent=4))

In [ ]:
#Creating a dataframe with each row having a tag per artist that we can then pivot to create a final tags table

df_tags = pd.DataFrame(rows)

df_tags

In [ ]:
df_tags.shape

In [ ]:
#1464 unique tags for just 1000 artists is fairly high, we can later filter these to take the more common tags

df_tags['tag'].nunique()

In [ ]:
#Taking the 60 most commonly occuring tags and creating a df of those to use later

df_final_tags = df_tags['tag'].value_counts().head(60).to_frame()

df_final_tags.reset_index(inplace=True)

df_final_tags.drop(columns=['count'], inplace=True)

df_final_tags

In [ ]:
#filtering the earlier df we made to include only the 60 tags from above

df_tags = df_tags[df_tags['tag'].isin(df_final_tags['tag'])]

df_tags.reset_index(inplace = True)

df_tags

In [ ]:
#Pivoting so that each tag is a column and each index (i.e. unique row) is an artist

df_artist_tags = df_tags.pivot_table(index='artist', columns='tag', values='count', aggfunc='sum', fill_value=0)

pd.set_option('display.max_columns', None)

df_artist_tags

In [ ]:
#From the API's artist.getSimilar we already know that these artists were grouped together
#Let's see what their tags look like in our df

pd.set_option('display.max_columns', None)

df_artist_tags.loc[['Keane','OneRepublic','Coldplay']]

In [ ]:
#Normalising the data so that higher number of tags don't dominate the model but the ratio between number of tags is maintained

from sklearn.preprocessing import normalize

df_artist_tags_norm = normalize(df_artist_tags, norm='l2')

df_artist_tags_norm

In [ ]:
df_artist_tags_norm.shape

In [ ]:
#Ensuring the df_artists dataframe we created earlier is filtered to only the ones who have our post-filtering tags

df_artists = df_artists[df_artists['name'].isin(df_artist_tags.index)]

### **PART 2 - CREATING THE CLUSTERS FOR ARTISTS THAT WILL BE USED TO RECOMMEND SIMILAR ARTISTS** ###

In [ ]:
#Before we run the actual code, how many clusters should be create?

inertias = []
k_range = range(1, 31)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(df_artist_tags_norm)
    inertias.append(km.inertia_)

In [ ]:
#Lowest intertia around 30 clusters, so we can go with that
#We did try with 20 clusters earlier but didn't get great results (very different artists in the same cluster)

import matplotlib.pyplot as plt

plt.plot(k_range, inertias, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
#Running the code to create the actual cluster based on elbow method above

kmeans = KMeans(n_clusters=30, random_state=42)
kmeans.fit(df_artist_tags_norm)

In [ ]:
df_artists['cluster'] = kmeans.labels_

In [ ]:
#Just checking which artists are in cluster 0

df_artists[df_artists['cluster'] == 0]['name'].values

In [ ]:
#Checking if there are artists with many 'blank' columns out of a total of 60.
#Seems like there are many artists with upto 59 'blank' tags/no tags from our list which makes clustering them hard

(df_artist_tags == 0).sum(axis=1).sort_values(ascending=False).head(20)

In [ ]:
#Checking how many artists correspond to the number of tags. For example, 12 artists have only 1 tag etc.

tag_counts = (df_artist_tags > 0).sum(axis=1)
print(tag_counts.value_counts().sort_index())

In [ ]:
#filtering the artists even further to only get those who have at least 5 tags

artists_to_keep = tag_counts[tag_counts >=5].index

df_artist_tags = df_artist_tags.loc[artists_to_keep]

df_artist_tags

In [ ]:
#normalizing the new df, which is filtered down to 810 artists

df_artist_tags_norm = normalize(df_artist_tags, norm='l2')

df_artist_tags_norm

In [ ]:
#Creating the new clusters (30) 

kmeans = KMeans(n_clusters=30, random_state=42)
kmeans.fit(df_artist_tags_norm)

In [ ]:
df_artists_filtered = df_artists[df_artists['name'].isin(df_artist_tags.index)]
df_artists_filtered = df_artists_filtered.reset_index(drop=True)
df_artists_filtered['cluster'] = kmeans.labels_

In [ ]:
df_artists_filtered[df_artists_filtered['cluster'] == 0]['name'].values

In [ ]:
#Checking what a few clusters look like

for i in range(5):
    print(f"Cluster {i}:")
    print(df_artists_filtered[df_artists_filtered['cluster'] == i]['name'].values)
    print()

In [ ]:
#Based on cosine_similarity we will recommend the closest 'n' artists to whatever the user inputs in the function

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_similar(artist_name, n=10):
    # find the artist's cluster
    cluster = df_artists_filtered[df_artists_filtered['name'] == artist_name]['cluster'].values[0]
    
    # get all artists in that cluster
    cluster_artists = df_artists_filtered[df_artists_filtered['cluster'] == cluster]['name'].values
    
    # get their tag vectors
    cluster_vectors = df_artist_tags.loc[cluster_artists]
    artist_vector = df_artist_tags.loc[artist_name]
    
    # calculate similarity
    similarities = cosine_similarity([artist_vector], cluster_vectors)[0]
    
    # sort and return top n (excluding the artist themselves)
    similar_indices = similarities.argsort()[::-1][1:n+1]
    return list(cluster_artists[similar_indices])

In [ ]:
recommend_similar('Coldplay', n=3)

In [ ]:
recommend_similar('Linkin Park', n = 10)

### **PART 3 - VALIDATING OUR RECOMMENDATION SYSTEM AGAINST LASTFM'S ARTISTS.GETSIMILAR RECOMMENDATION SYSTEM** ###

In [ ]:
all_recommendations = []

for artist in df_artists_filtered['name']:
    try:
        recs = recommend_similar(artist, n=5)
        all_recommendations.append({'artist': artist, 'recs': recs})
    except:
        all_recommendations.append({'artist': artist, 'recs': []})

df_model_recs = pd.DataFrame(all_recommendations)

In [ ]:
df_model_recs

In [ ]:
import time

lastfm_recs = []

for artist in df_artists_filtered['name']:
    params = {
        "method": "artist.getSimilar",
        "limit": 10,
        "artist": artist,
        "autocorrect": 1,
        "api_key": api_key,
        "format":"json"
    }
    try:
        response = requests.get(url, params)
        data = response.json()
        similar = data['similarartists']['artist']
        names = [a['name'] for a in similar]
        lastfm_recs.append({'artist': artist, 'lastfm_recs': names})
    except:
        lastfm_recs.append({'artist': artist, 'lastfm_recs': []})
    
    time.sleep(0.2)

df_lastfm_recs = pd.DataFrame(lastfm_recs)

In [ ]:
df_lastfm_recs

In [ ]:
all_artists = set(df_artists_filtered['name'])

df_validation['overlap_fair'] = df_validation.apply(
    lambda row: len(set(row['recs']) & set(row['lastfm_recs'])) / 
    max(len([x for x in row['lastfm_recs'] if x in all_artists]), 1),
    axis=1
)

print(df_validation['overlap_fair'].mean())